In [2]:
from tqdm import tqdm
import pandas as pd
import spacy
import re

## 1. run NER on corpus

In [ ]:
# %python -m spacy download fr_core_news_lg

In [3]:
pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [9]:
presse = pd.read_csv("/Users/pol/Downloads/press_corpus.csv")
presse

,source,article_text,article_id
0,telerama,Serge Reggiani - Succès & confidences .\n Une ...,1
1,telerama,"Désolés, Warren .\n C'est une maigre consolati...",2
2,telerama,Adam Green - Friends of mine .\n Dire qu'on a ...,3
3,telerama,Cesaria Evora - Voz d'amor .\n Dès les premièr...,4
4,telerama,Herman Düne - Mas cambios et Mash Concrete Met...,5
...,...,...,...
52476,lemonde,"Rap, jazz, rock, classique, chanson… Les album...",52477
52477,lemonde,Que sait-on vraiment des surrisques de contami...,52478
52478,lemonde,"Rencontre avec Riccardo Muti, maestro à vie .\...",52479
52479,lemonde,Ils sont passés chez Colors : la sélection mus...,52480


In [19]:
nlp = spacy.load('fr_core_news_lg')

In [ ]:
df = presse

# create spacy doc object with article_text, keep article id
# parallelize
docs = nlp.pipe(
    zip(df["article_text"], df["article_id"]),
    as_tuples=True,
    disable=['tok2vec', 'parser', 'lemmatizer'],
    batch_size=8,
    n_process=4
)

result = []

# for all ents that are not LOC, extract ents contained in each article
with tqdm(total=len(df)) as pbar:
    for doc, idx in docs: 
        for ent in doc.ents:
            if  ent.label_ != 'LOC':
                result.append((idx, ent.text, ent.label_))
            
        pbar.update(1)
        
ner_df = pd.DataFrame(result, columns=['article_id', 'ent_name', 'ner_type'])
ner_df.to_csv("/Users/pol/Downloads/ner_df_raw.csv", 
                      sep=";",
                      encoding="utf-8-sig")

100%|██████████| 52481/52481 [20:10<00:00, 43.37it/s] 


## 2. add counts to entities 

- in total
- by newspaper

In [ ]:
ner_df = pd.read_csv("/Users/pol/Downloads/ner_df_raw.csv",
                     sep=";",
                      encoding="utf-8-sig")

ner_df = ner_df.drop(columns="Unnamed: 0")
ner_df = ner_df.dropna(subset="ent_name")

ner_df

,article_id,name,ner_type,ent_name
0,1,Serge Reggiani - Succès & confidences,PER,Serge Reggiani - Succès & confidences
1,1,Reggiani,ORG,Reggiani
2,1,Brel,PER,Brel
3,1,Barbara,PER,Barbara
4,1,Ventura,PER,Ventura
...,...,...,...,...
1470598,52481,Moussorgsky,PER,Moussorgsky
1470599,52481,Galina Vichnevskaïa,PER,Galina Vichnevskaïa
1470600,52481,Arturo Benedetti Michelangeli,PER,Arturo Benedetti Michelangeli
1470601,52481,Ballade no 1,MISC,Ballade no 1


In [7]:
# solve tiny issue with names starting with - (that lead to csv encoding error)
def remove_tiretdusix(string):
    string = re.sub(pattern="^-", repl="", string=string)
    string = string.strip()
    return string

ner_df["ent_name"] = ner_df["ent_name"].apply(remove_tiretdusix)

In [10]:
# count occurrences of each name
ner_df["name_count"] = ner_df["ent_name"].map(ner_df["ent_name"].value_counts()) # in total
# ner_df["article_count"] = ner_df.groupby("name")["article_id"].transform("nunique") # in N articles

# join source again to prepare decomposition into media outlets
ner_df = ner_df.merge(
    presse[["article_id", "source"]],
    on="article_id",
    how="left"
)

ner_df

,article_id,name,ner_type,ent_name,name_count,source
0,1,Serge Reggiani - Succès & confidences,PER,Serge Reggiani - Succès & confidences,1,telerama
1,1,Reggiani,ORG,Reggiani,36,telerama
2,1,Brel,PER,Brel,337,telerama
3,1,Barbara,PER,Barbara,664,telerama
4,1,Ventura,PER,Ventura,16,telerama
...,...,...,...,...,...,...
1470597,52481,Moussorgsky,PER,Moussorgsky,4,lemonde
1470598,52481,Galina Vichnevskaïa,PER,Galina Vichnevskaïa,10,lemonde
1470599,52481,Arturo Benedetti Michelangeli,PER,Arturo Benedetti Michelangeli,9,lemonde
1470600,52481,Ballade no 1,MISC,Ballade no 1,2,lemonde


In [11]:
# decompose name and article count into media outlet
source_stats = (
    ner_df.groupby(["ent_name", "source"])
    .agg(
        name_count=("ent_name", "size"),
        #article_count=("article_id", "nunique")
    )
)

wide = source_stats.unstack(fill_value=0)
wide.columns = [f"{metric}_{source}" for metric, source in wide.columns]
wide = wide.reset_index()
wide

,ent_name,name_count_lefigaro,name_count_lemonde,name_count_liberation,name_count_telerama
0,!\n\n- Ah mon Olive,0,0,1,0
1,!\n\n- Calme-toi,0,0,1,0
2,!\n\n- Mmm,0,0,1,0
3,!\n\nA,0,0,2,0
4,!\n\nA la Libération,0,0,1,0
...,...,...,...,...,...
408595,🌳\n\nAucun,0,0,1,0
408596,👑✌,0,0,1,0
408597,👴❤👵,0,0,1,0
408598,😡😡😡@philharmonie,0,0,1,0


In [12]:
# df of unique extracted entities
extracted_ents = (
    ner_df
    .groupby("ent_name")
    .agg({"article_id": "first",
          "name_count": "first",
          #"article_count": "first"
          })
    .sort_values(by='name_count', ascending=False)
    .reset_index()
    .merge(wide,
           on="ent_name",
           how="left")
)

extracted_ents["name_id"] = extracted_ents.index + 1
extracted_ents

,ent_name,article_id,name_count,name_count_lefigaro,name_count_lemonde,name_count_liberation,name_count_telerama,name_id
0,Ça,9,4727,923,460,3088,256,1
1,CD,2,3758,240,627,1715,1176,2
2,Libération,870,3349,89,35,3211,14,3
3,Figaro,1786,2749,2651,10,80,8,4
4,Internet,40,2374,312,313,1668,81,5
...,...,...,...,...,...,...,...,...
408595,J'adorais ça,4632,1,0,0,0,1,408596
408596,J'accuse...!,17459,1,1,0,0,0,408597
408597,J'Adore voile de Parfum de Dior,34466,1,0,0,1,0,408598
408598,J'Accuse\n\nMeilleure musique,45584,1,0,0,1,0,408599


In [14]:
# save
extracted_ents.to_csv("/Users/pol/Downloads/extracted_ents_1203.csv", 
                      sep=";",
                      encoding="utf-8-sig")

## 3. append entities to press_corpus

In [38]:
# press_corpus with column listing all entities for each text
article_text = presse[["article_id", "article_text"]]
article_text

,article_id,article_text
0,1,Serge Reggiani - Succès & confidences .\n Une ...
1,2,"Désolés, Warren .\n C'est une maigre consolati..."
2,3,Adam Green - Friends of mine .\n Dire qu'on a ...
3,4,Cesaria Evora - Voz d'amor .\n Dès les premièr...
4,5,Herman Düne - Mas cambios et Mash Concrete Met...
...,...,...
52476,52477,"Rap, jazz, rock, classique, chanson… Les album..."
52477,52478,Que sait-on vraiment des surrisques de contami...
52478,52479,"Rencontre avec Riccardo Muti, maestro à vie .\..."
52479,52480,Ils sont passés chez Colors : la sélection mus...


In [39]:
press_corpus_ents = (
    ner_df
    .groupby("article_id").agg({
            "name": list,
            #"name_count": "first",
            #"article_count": "first"
        })
    .join(article_text, on="article_id")
)

press_corpus_ents = press_corpus_ents[["article_id", "article_text", "name"]]
press_corpus_ents

,article_id,article_text,name
article_id,,,
1,1,"Désolés, Warren .\n C'est une maigre consolati...","[Serge Reggiani - Succès & confidences, Reggia..."
2,2,Adam Green - Friends of mine .\n Dire qu'on a ...,"[Warren, Warren Zevon, I'll sleep when I'm dea..."
3,3,Cesaria Evora - Voz d'amor .\n Dès les premièr...,"[Adam Green - Friends of mine, The Kills, Gree..."
4,4,Herman Düne - Mas cambios et Mash Concrete Met...,"[gorgée, Voz, Voix de l'amour, Velocidade, Ces..."
5,5,"Albin de la Simone .\n """" Du Souchon dans la v...","[Herman Düne - Mas cambios, Mash Concrete Meta..."
...,...,...,...
52477,52477,Que sait-on vraiment des surrisques de contami...,"[Rap, Monde, An Evening with Silk Sonic, Silk ..."
52478,52478,"Rencontre avec Riccardo Muti, maestro à vie .\...","[Covid-19, Omicron, Jean Castex, Covid-19, Clu..."
52479,52479,Ils sont passés chez Colors : la sélection mus...,"[Riccardo Muti, Riccardo Muti, Cristina, Ricca..."


In [40]:
press_corpus_ents.to_csv("/Users/pol/Downloads/press_corpus_ents_1103.csv", 
                   sep=";",
                   encoding="utf-8-sig")
press_corpus_ents

,article_id,article_text,name
article_id,,,
1,1,"Désolés, Warren .\n C'est une maigre consolati...","[Serge Reggiani - Succès & confidences, Reggia..."
2,2,Adam Green - Friends of mine .\n Dire qu'on a ...,"[Warren, Warren Zevon, I'll sleep when I'm dea..."
3,3,Cesaria Evora - Voz d'amor .\n Dès les premièr...,"[Adam Green - Friends of mine, The Kills, Gree..."
4,4,Herman Düne - Mas cambios et Mash Concrete Met...,"[gorgée, Voz, Voix de l'amour, Velocidade, Ces..."
5,5,"Albin de la Simone .\n """" Du Souchon dans la v...","[Herman Düne - Mas cambios, Mash Concrete Meta..."
...,...,...,...
52477,52477,Que sait-on vraiment des surrisques de contami...,"[Rap, Monde, An Evening with Silk Sonic, Silk ..."
52478,52478,"Rencontre avec Riccardo Muti, maestro à vie .\...","[Covid-19, Omicron, Jean Castex, Covid-19, Clu..."
52479,52479,Ils sont passés chez Colors : la sélection mus...,"[Riccardo Muti, Riccardo Muti, Cristina, Ricca..."
